In [67]:
from deck import full_euchre_deck
from dealer import Dealer
from n_game_sim import n_game_sim
from n_play_round import round1, n_play_round, next_round
from n_branches import array_set_difference, n_find_winner, array_set_difference
from tree_search import find_best_opener, n_tree_sim, four_trick_sim, trick_played #, r2_best_response
import numpy as np
from numba import njit

In [68]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands5 = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands5

array([[[  0, 140],
        [ 11,   0],
        [-13,   0],
        [ 12,   0],
        [-12,   0]],

       [[  0, 120],
        [  0, 135],
        [-14,   0],
        [  0, 100],
        [  0,  -9]],

       [[-11,   0],
        [  0, 130],
        [  0,  90],
        [  9,   0],
        [  0, -14]],

       [[ -9,   0],
        [ 10,   0],
        [  0, 110],
        [ 14,   0],
        [  0, -13]]])

In [69]:
best_opener = find_best_opener(hands=hands5, player=0, lead=0, tricks=5, sim_func=n_tree_sim)

[0.44401544 0.32111493 0.23294509 0.31498471 0.23352166]


In [70]:
r2_leads, r2_hands = round1(hands_dealt=hands5, chosen_card=best_opener, leader=0)

In [71]:
r2_leads

array([0])

In [72]:
# find the best response by iterating through each possible trick from the perspective of the opponent

# need to make this into a function as well

eval_position=0

r1_response_res = np.zeros(r2_leads.shape, dtype=np.float64)

for w in range(r2_leads.shape[0]):


    r3_leads, r3_hands, r2_score = next_round(
        current_hands=np.array([r2_hands[w]]),
        leads=np.array([r2_leads[w]]),
        game_round=2,
        game_score=np.array([r2_leads[w]]).reshape(-1, 1),
    )
    r4_leads, r4_hands, r3_score = next_round(
        current_hands=r3_hands, leads=r3_leads, game_round=3, game_score=r2_score
    )
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    if eval_position % 2 == 0:
        for i in range(len(results)):
            meta_results[i] = np.sum(results[i]%2)<3
    else:
        for i in range(len(results)):
            meta_results[i] = np.sum(results[i]%2)>=3

    r1_response_res[w] = np.mean(meta_results)
best_response = np.argmin(r1_response_res)
print(np.argmin(r1_response_res))

0


In [73]:
trick_played(hands5, r2_hands[best_response])

array([[  0, 140],
       [  0, 100],
       [  0,  90],
       [  0, 110]])

In [74]:
r2_leads[best_response]

np.int64(0)

In [75]:
r2_winner = r2_leads[best_response]

In [76]:
r2_optimal =  r2_hands[best_response]

In [77]:
r2_optimal[0]

array([[ 11,   0],
       [-13,   0],
       [ 12,   0],
       [-12,   0]])

In [78]:

r2_best_opener = find_best_opener(hands=r2_optimal, player=0, lead=r2_winner, tricks=4, sim_func=four_trick_sim)

[0.43846154 0.4375     0.45098039 0.45454545]
